# Parquet: Fast, Columnar Storage for Engineering Data

## The Core Question

You have a CFD simulation with 250K cell points and 9 fields. You save it as CSV. Next time you load it, it takes seconds. You only need 3 fields. Why did you load 6 you don't use?

**Parquet** is a columnar storage format designed for analytics: read only the columns you need, compress better than CSV, and handle typed numeric data natively. For simulation data, it's typically **10-50x faster to read** than CSV.

## Why Parquet Over CSV?

| Feature | CSV | Parquet |
|---------|-----|---------|
| Storage | Row-oriented | Column-oriented |
| Types | All strings | Native int/float/datetime |
| Compression | None (or gzip whole file) | Per-column (snappy, zstd) |
| Read only some columns | No (must parse all) | Yes (skip columns) |
| 250K rows, 9 cols | ~30 MB | ~3-8 MB |

## The Workflow

```
Simulation output (numpy arrays / pandas DataFrame)
        |
        v
  Write to Parquet (one file per case or partitioned by run)
        |
        v
  Read back only needed columns/rows (predicate pushdown)
        |
        v
  Process / plot / compare
```

## Setup


In [ ]:
import numpy as np
import pandas as pd
import time
import os

pd.set_option('display.max_columns', 20)
pd.set_option('display.width', 120)


## 1. Generate Synthetic CFD Data

**Concept**: Simulate a 2D velocity field on a 250K-point grid with 9 fields. This is small by CFD standards (real cases have 10M+ cells), but enough to demonstrate the speed difference.


In [ ]:
np.random.seed(42)
n_cells = 250_000

x = np.random.uniform(0, 2, n_cells)
y = np.random.uniform(0, 0.5, n_cells)
u = 1.0 + 0.3 * np.sin(np.pi * x)
v = 0.05 * np.cos(2 * np.pi * x)
w = 0.01 * np.random.randn(n_cells)
p = 101325 - 500 * x
T = 300 + 10 * np.exp(-((x-0.5)**2 + (y-0.25)**2) * 20)
k = 0.01 * np.abs(u - 1.0)
omega = 5.0 + 2.0 * x

df = pd.DataFrame({'x': x, 'y': y, 'u': u, 'v': v, 'w': w, 'p': p, 'T': T, 'k': k, 'omega': omega})
print(f'DataFrame: {df.shape[0]:,} rows x {df.shape[1]} cols')
print(f'Memory (in-RAM): {df.memory_usage(deep=True).sum() / 1e6:.1f} MB')
df.head(3)


## 2. Write and Compare: CSV vs Parquet

**Concept**: Write the same data to both formats, compare file sizes and write times.


In [ ]:
for f in ['demo_data.csv', 'demo_data.parquet']:
    if os.path.exists(f):
        os.remove(f)

t0 = time.perf_counter()
df.to_csv('demo_data.csv', index=False)
csv_write_time = time.perf_counter() - t0

t0 = time.perf_counter()
df.to_parquet('demo_data.parquet', index=False, engine='pyarrow')
pq_write_time = time.perf_counter() - t0

csv_size = os.path.getsize('demo_data.csv') / 1e6
pq_size = os.path.getsize('demo_data.parquet') / 1e6

print('File sizes:')
print(f'  CSV:     {csv_size:.1f} MB')
print(f'  Parquet: {pq_size:.1f} MB ({csv_size/pq_size:.1f}x smaller)')
print(f'\nWrite times:')
print(f'  CSV:     {csv_write_time*1000:.0f} ms')
print(f'  Parquet: {pq_write_time*1000:.0f} ms')


## 3. Read Performance: Full vs Partial

**The key advantage**: Parquet reads *only the columns you need*. For CFD post-processing where you often want just `x, u, p` out of 9 fields, this is huge.


In [ ]:
t0 = time.perf_counter()
df_csv_full = pd.read_csv('demo_data.csv')
csv_full_time = time.perf_counter() - t0

t0 = time.perf_counter()
df_pq_full = pd.read_parquet('demo_data.parquet')
pq_full_time = time.perf_counter() - t0

t0 = time.perf_counter()
df_pq_cols = pd.read_parquet('demo_data.parquet', columns=['x', 'u', 'p'])
pq_cols_time = time.perf_counter() - t0

t0 = time.perf_counter()
df_csv_cols = pd.read_csv('demo_data.csv')[['x', 'u', 'p']]
csv_cols_time = time.perf_counter() - t0

print('Read performance (250K rows, 9 cols):')
print(f'  Full CSV:        {csv_full_time*1000:7.1f} ms')
print(f'  Full Parquet:    {pq_full_time*1000:7.1f} ms  ({csv_full_time/pq_full_time:.1f}x faster)')
print(f'  3 cols from CSV: {csv_cols_time*1000:7.1f} ms  (still reads all!)')
print(f'  3 cols from Pq:  {pq_cols_time*1000:7.1f} ms  ({csv_cols_time/pq_cols_time:.1f}x faster)')


## 4. Row Filtering with PyArrow

**Concept**: You want only cells near the leading edge (x < 0.1). PyArrow's predicate pushdown skips row groups at the storage level.


In [ ]:
import pyarrow.parquet as pq

# Write with row groups
df.to_parquet('demo_data_rg.parquet', index=False, row_group_size=50_000)

# Read with filter (predicate pushdown)
t0 = time.perf_counter()
table = pq.read_table('demo_data_rg.parquet', filters=[('x', '<', 0.1)])
pq_filter_time = time.perf_counter() - t0
le_filtered = table.to_pandas()

# Same from CSV (must read all, then filter)
t0 = time.perf_counter()
df_le_csv = pd.read_csv('demo_data.csv')
csv_le_time = time.perf_counter() - t0
le_csv = df_le_csv[df_le_csv['x'] < 0.1]

print(f'Leading edge (x < 0.1): {len(le_filtered):,} cells')
print(f'  Parquet + filter: {pq_filter_time*1000:.1f} ms')
print(f'  CSV + filter:     {csv_le_time*1000:.1f} ms')
print(f'  Speedup: {csv_le_time/pq_filter_time:.1f}x')


## 5. Multi-Case: Partitioned Parquet

**Concept**: You ran 5 alpha angles. Write a **partitioned dataset**. Pandas reads it as one logical table with a free `alpha` column for grouping.


In [ ]:
import shutil
if os.path.exists('multi_case'):
    shutil.rmtree('multi_case', ignore_errors=True)

alphas = [5, 10, 15, 20, 25]
for alpha in alphas:
    u_case = 1.0 + 0.1 * (alpha / 10) * np.sin(np.pi * x)
    p_case = 101325 - 800 * x * (alpha / 10)**0.5
    df_case = df[['x', 'y', 'u', 'v', 'p']].copy()
    df_case['u'] = u_case
    df_case['p'] = p_case
    df_case['alpha'] = alpha
    df_case.to_parquet('multi_case', columns=['x', 'y', 'u', 'v', 'p', 'alpha'],
                       index=False, partition_cols=['alpha'])

# Read back: one call, all cases
df_all = pd.read_parquet('multi_case')
print(f'Partitioned read: {df_all.shape[0]:,} rows, {df_all.shape[1]} cols')
print(f'Alphas: {sorted(df_all["alpha"].unique().tolist())}')

summary = df_all.groupby('alpha')[['p', 'u']].mean()
print('\nPer-case summary:')
print(summary)


## 6. Practical Tips

| Situation | Recommendation |
|-----------|---------------|
| Single CFD case, post-process once | Parquet single file |
| 50+ cases, compare across alphas | Partitioned by case/alpha |
| Time-resolved (transient) | Partition by time step |
| Shared with non-Python tools (Java, Spark, R) | Parquet (universal) |
| Very small data (<10K rows) | CSV is fine |
| Streaming from HPC multi-rank | Write Parquet per rank, merge later |

**Rule of thumb**: If your CSV is >10 MB, switch to Parquet. Migration is one line: `df.to_parquet('out.parquet')` instead of `df.to_csv('out.csv')`.


## Exercises

1. **Real data**: Take a CSV from your last CFD post-processing. Convert to Parquet, time the read, verify numbers match.

2. **Time series**: Generate 100 time steps of a 1D flow. Write as partitioned Parquet. Read only steps 40-60.

3. **Compression**: Compare `compression='snappy'` vs `'zstd'` vs `None`. What's the size/speed tradeoff?

4. **Schema evolution**: Write a Parquet file, add a column to the DataFrame, try to read the old file. What happens?

5. **Benchmark**: Time CSV vs Parquet on your actual largest dataset. Report the speedup in one line.
